# Prepare Data for Predictive Model
In this notebook we will prepare the data for the predictive model.


In [1]:
from pathlib import Path

import duckdb as db
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12

# Detect project root robustly.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
FIGURES_DIR = OUTPUT_DIR / "figures"
TABLES_DIR = OUTPUT_DIR / "tables"

PARQUET_PATH = PROCESSED_DIR / "crimes_clean_dedup_all_years.parquet"
PARQUET_SQL_PATH = PARQUET_PATH.as_posix()

# true to output the figures
EXPORT_OUTPUTS = False

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Crime parquet path: {PARQUET_PATH}")
print(f"Parquet exists: {PARQUET_PATH.exists()}")

if not PARQUET_PATH.exists():
    raise FileNotFoundError(
        "Could not find crimes_clean_dedup_all_years.parquet. "
        "Expected it under data/processed/."
    )


Project root: /home/rovsi/Courses/Multidisciplinary_CBL
Crime parquet path: /home/rovsi/Courses/Multidisciplinary_CBL/data/processed/crimes_clean_dedup_all_years.parquet
Parquet exists: True


### Filtering Covid and Inital years:
As seen in EDA 2010 and 2011 has abnormal amount of "other crime". 
Furthermore, our research supports covid-era to be excluded since it does not represent the normal trend.

In [2]:
#Data Overview
sample_rows = db.sql(f"""
    SELECT *
    FROM read_parquet('{PARQUET_SQL_PATH}') AS crimes
    ORDER BY Month asc
    LIMIT 5
""").df()

sample_rows

,Month,Falls within,Longitude,Latitude,LSOA code,LSOA name,Crime type
0,2010-12,Avon and Somerset Constabulary,-2.499910,51.413623,E01014400,Bath and North East Somerset 001B,Other crime
1,2010-12,Avon and Somerset Constabulary,-2.357276,51.390542,E01014466,Bath and North East Somerset 006B,Other crime
2,2010-12,Avon and Somerset Constabulary,-2.355926,51.383272,E01014371,Bath and North East Somerset 007B,Vehicle crime
3,2010-12,Avon and Somerset Constabulary,-2.383849,51.381799,E01014475,Bath and North East Somerset 013C,Anti-social behaviour
4,2010-12,Avon and Somerset Constabulary,-2.385498,51.371246,E01014476,Bath and North East Somerset 013D,Anti-social behaviour


In [3]:
#We only keep the crimes happened after 2012-01, and outside the covid-era ()
#Save it under data/processed/ for future.

OUTPUT_PATH = PROCESSED_DIR / "crimes_filtered_for_model.parquet"
con = db.connect()

con.execute(f"""
    COPY (
        SELECT *
        FROM read_parquet('{PARQUET_PATH.as_posix()}') AS crimes
        WHERE
            Month >= '2012-01'
            AND (
                Month < '2020-03'
                OR Month > '2021-03'
            )
    )
    TO '{OUTPUT_PATH.as_posix()}'
    (FORMAT parquet, COMPRESSION zstd);
""")

print(f"Saved filtered data to: {OUTPUT_PATH}")
print(f"Output exists: {OUTPUT_PATH.exists()}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Saved filtered data to: /home/rovsi/Courses/Multidisciplinary_CBL/data/processed/crimes_filtered_for_model.parquet
Output exists: True


In [4]:
#Test the above:
PARQUET_PATH_FILTERED = PROCESSED_DIR / "crimes_filtered_for_model.parquet"
PARQUET_SQL_PATH_FILTERED = PARQUET_PATH_FILTERED.as_posix()

sample_rows_filtered = db.sql(f"""
    SELECT *
    FROM read_parquet('{PARQUET_SQL_PATH_FILTERED}') AS crimes
    ORDER BY Month asc
    LIMIT 5
""").df()
sample_rows_filtered


,Month,Falls within,Longitude,Latitude,LSOA code,LSOA name,Crime type
0,2012-01,Avon and Somerset Constabulary,-2.500575,51.411345,E01014403,Bath and North East Somerset 002B,Violent crime
1,2012-01,Avon and Somerset Constabulary,-2.508310,51.407410,E01014404,Bath and North East Somerset 002C,Anti-social behaviour
2,2012-01,Avon and Somerset Constabulary,-2.382552,51.393465,E01014479,Bath and North East Somerset 005C,Anti-social behaviour
3,2012-01,Avon and Somerset Constabulary,-2.360893,51.382546,E01014370,Bath and North East Somerset 007A,Other theft
4,2012-01,Avon and Somerset Constabulary,-2.356174,51.379962,E01014371,Bath and North East Somerset 007B,Other crime


In [6]:
#Confirm that covid data is not there
covid_data_check = db.sql(f"""
    SELECT Month, COUNT(*) AS crime_count
    FROM read_parquet('{PARQUET_SQL_PATH_FILTERED}') AS crimes
    WHERE Month >= '2020-01' AND Month < '2021-06'
    GROUP BY Month
    ORDER BY Month asc
    LIMIT 10
""").df()
covid_data_check

,Month,crime_count
0,2020-01,360560
1,2020-02,341514
2,2021-04,367904
3,2021-05,374681
